# Module 30: Debugging PyTorch — A Systematic Guide

**Time**: ~2 hours | **Prerequisites**: Modules 07, 08 | **GPU**: Not required

---

## What You'll Learn

- The systematic debugging mindset: reproduce → isolate → identify → fix
- Anomaly detection for catching NaN in backward pass
- NaN/Inf detection with hooks
- Gradient flow checking (vanishing/exploding)
- Shape debugging strategies
- torch.compile debugging (graph breaks, recompilations)
- Memory leak detection
- Reproducibility setup for bug reports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch._dynamo as dynamo
import numpy as np
import random

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. The Debugging Mindset

### The Protocol

```
Reproduce → Isolate → Identify → Fix → Verify
```

**Golden rule**: Always create a minimal reproducer first. If you can't reproduce it in <20 lines, you don't understand it yet.

### Common debugging anti-patterns:
- Randomly changing hyperparameters ("maybe a smaller LR will fix it")
- Adding more code before understanding the bug
- Ignoring warnings and deprecation notices
- Not checking tensor values at each step

## 2. Anomaly Detection

`torch.autograd.detect_anomaly()` catches NaN in backward, in-place ops on grad-requiring tensors, and double-backward issues.

In [ ]:
# Anomaly detection: catches NaN in backward pass

class ProblematicModel(nn.Module):
    def forward(self, x):
        # log(0) = -inf, whose gradient is NaN
        return torch.log(x)

model = ProblematicModel()

# Input with a zero — will produce NaN gradient
x = torch.tensor([1.0, 0.5, 0.0], requires_grad=True)

print("Without detect_anomaly: NaN propagates silently")
out = model(x)
print(f"  Output: {out}")
# backward would produce NaN for the 0.0 element

print("\nWith detect_anomaly: catches the NaN")
try:
    with torch.autograd.detect_anomaly():
        out = model(x)
        out.sum().backward()
except RuntimeError as e:
    print(f"  Caught: {str(e)[:120]}...")

print("\nFix: clamp input to avoid log(0)")
x_safe = x.detach().clamp(min=1e-8).requires_grad_(True)
with torch.autograd.detect_anomaly():
    out = model(x_safe)
    out.sum().backward()
print(f"  Gradient (safe): {x_safe.grad}")

In [ ]:
# Performance cost of detect_anomaly
import time

model = nn.Sequential(nn.Linear(100, 100), nn.ReLU(), nn.Linear(100, 10))
x = torch.randn(32, 100)

# Without anomaly detection
start = time.time()
for _ in range(1000):
    out = model(x).sum()
    out.backward()
    model.zero_grad()
normal_time = time.time() - start

# With anomaly detection
start = time.time()
with torch.autograd.detect_anomaly():
    for _ in range(1000):
        out = model(x).sum()
        out.backward()
        model.zero_grad()
anomaly_time = time.time() - start

print(f"Normal:  {normal_time:.3f}s")
print(f"Anomaly: {anomaly_time:.3f}s")
print(f"Overhead: {anomaly_time/normal_time:.1f}x slower")
print("\n→ Only enable during debugging!")

## 3. NaN/Inf Detection

Register hooks to automatically detect NaN/Inf values as they appear in the forward and backward pass.

In [ ]:
# NaN/Inf detection hooks

def make_nan_hook(name):
    """Create a forward hook that checks for NaN/Inf."""
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            if torch.isnan(output).any():
                print(f"  ⚠ NaN in {name}: {torch.isnan(output).sum().item()} values")
            if torch.isinf(output).any():
                print(f"  ⚠ Inf in {name}: {torch.isinf(output).sum().item()} values")
    return hook

# Model that produces NaN
class LeakyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
    
    def forward(self, x):
        x = self.linear(x)
        x = torch.log(x)  # Negative values → NaN!
        return x

model = LeakyModel()

# Register hooks on all modules
for name, module in model.named_modules():
    module.register_forward_hook(make_nan_hook(name))

print("Running forward pass with NaN-producing model:")
with torch.no_grad():
    x = torch.randn(2, 10)
    output = model(x)
    print(f"\n  Output has NaN: {torch.isnan(output).any().item()}")
    print(f"  Output has Inf: {torch.isinf(output).any().item()}")

In [ ]:
# Common NaN causes and fixes

print("Common NaN/Inf causes:\n")

# 1. log(0)
x = torch.tensor([0.5, 0.0, -0.1])
print(f"1. log(x) where x={x}: {torch.log(x)}")
print(f"   Fix: log(clamp(x, min=1e-8)): {torch.log(x.clamp(min=1e-8))}")

# 2. Division by zero
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([1.0, 0.0, 2.0])
print(f"\n2. a/b where b has zero: {a/b}")
print(f"   Fix: a/(b + 1e-8): {a/(b + 1e-8)}")

# 3. sqrt(0) gradient
x = torch.tensor([4.0, 0.0, 1.0], requires_grad=True)
y = torch.sqrt(x)
y.sum().backward()
print(f"\n3. sqrt gradient at 0: {x.grad}")
print(f"   Fix: sqrt(x + eps)")

# 4. Softmax overflow
logits = torch.tensor([1000.0, 1.0, -1000.0])
print(f"\n4. exp(large): {torch.exp(logits)}")
print(f"   Fix: Use log_softmax (numerically stable)")
print(f"   log_softmax: {F.log_softmax(logits, dim=0)}")

## 4. Gradient Flow Checking

Verify gradients are flowing through all layers. Detect vanishing and exploding gradients.

In [ ]:
# Gradient flow checker

def check_gradient_flow(model):
    """Print gradient norms for each parameter."""
    print(f"{'Layer':<35} {'Grad Norm':<12} {'Status'}")
    print("-" * 60)
    for name, param in model.named_parameters():
        if param.grad is None:
            print(f"{name:<35} {'None':<12} NO GRADIENT")
        else:
            norm = param.grad.norm().item()
            if norm < 1e-7:
                status = "VANISHING!"
            elif norm > 100:
                status = "EXPLODING!"
            else:
                status = "OK"
            print(f"{name:<35} {norm:<12.6f} {status}")

# Deep network with Sigmoid (prone to vanishing gradients)
class DeepSigmoidNet(nn.Module):
    def __init__(self, depth=8):
        super().__init__()
        layers = []
        for i in range(depth):
            layers.append(nn.Linear(32, 32))
            layers.append(nn.Sigmoid())
        layers.append(nn.Linear(32, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

model = DeepSigmoidNet(depth=8)
x = torch.randn(4, 32)
loss = model(x).sum()
loss.backward()

print("Gradient flow in 8-layer Sigmoid network:")
print("(Early layers should show vanishing gradients)\n")
check_gradient_flow(model)

In [ ]:
# Fix: Replace Sigmoid with ReLU + use residual connections

class DeepReLUNet(nn.Module):
    def __init__(self, depth=8):
        super().__init__()
        layers = []
        for i in range(depth):
            layers.append(nn.Linear(32, 32))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(32, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

model_fixed = DeepReLUNet(depth=8)
x = torch.randn(4, 32)
loss = model_fixed(x).sum()
loss.backward()

print("Gradient flow with ReLU (much better!):\n")
check_gradient_flow(model_fixed)

## 5. Shape Debugging

Shape mismatches are the #1 most common error. Use systematic approaches to find them.

In [ ]:
# Shape logging hook

class ShapeTracer:
    """Trace tensor shapes through a model."""
    def __init__(self):
        self.traces = []
        self._handles = []
    
    def attach(self, model):
        for name, module in model.named_modules():
            if not list(module.children()):  # leaf only
                h = module.register_forward_hook(self._make_hook(name))
                self._handles.append(h)
    
    def _make_hook(self, name):
        def hook(mod, inp, out):
            in_shape = inp[0].shape if isinstance(inp[0], torch.Tensor) else '?'
            out_shape = out.shape if isinstance(out, torch.Tensor) else '?'
            self.traces.append((name, in_shape, out_shape))
        return hook
    
    def print_traces(self):
        print(f"{'Module':<25} {'Input Shape':<20} {'Output Shape'}")
        print("-" * 65)
        for name, in_s, out_s in self.traces:
            print(f"{name:<25} {str(in_s):<20} {out_s}")
        self.traces.clear()
    
    def remove(self):
        for h in self._handles:
            h.remove()

# Example model
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(32, 10),
)

tracer = ShapeTracer()
tracer.attach(model)

x = torch.randn(4, 3, 32, 32)
print(f"Input shape: {x.shape}\n")
_ = model(x)
tracer.print_traces()
tracer.remove()

In [ ]:
# Catch and diagnose shape mismatches

model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)

# Wrong input size
x_wrong = torch.randn(8, 100)  # Should be 784!

print("Diagnosing shape mismatch:")
try:
    model(x_wrong)
except RuntimeError as e:
    print(f"  Error: {e}")
    print(f"  Input shape: {x_wrong.shape}")
    print(f"  Expected: (batch, 784)")
    print(f"  Fix: Ensure your data pipeline produces the right shape")

## 6. Common Errors Quiz

For each error below, identify the cause and fix before looking at the solution.

---

**Error 1:**
```
RuntimeError: one of the variables needed for gradient computation has been 
modified by an inplace operation
```

<details><summary>Cause & Fix</summary>

**Cause**: Using in-place operations (like `x.add_()`, `x *= 2`) on tensors that are part of the computation graph.

**Fix**: Replace `x.add_(1)` with `x = x + 1`. Avoid in-place ops on tensors with `requires_grad=True`.
</details>

---

**Error 2:**
```
RuntimeError: Trying to backward through the graph a second time
```

<details><summary>Cause & Fix</summary>

**Cause**: Calling `.backward()` twice on the same graph without `retain_graph=True`.

**Fix**: Use `loss.backward(retain_graph=True)` if you need multiple backwards, or restructure to compute loss once.
</details>

---

**Error 3:**
```
RuntimeError: Expected all tensors to be on the same device, but found 
at least two devices, cuda:0 and cpu!
```

<details><summary>Cause & Fix</summary>

**Cause**: Model is on GPU but input/target is on CPU, or vice versa.

**Fix**: Move everything to the same device: `x = x.to(device)`, `target = target.to(device)`.
</details>

---

**Error 4:**
```
RuntimeError: CUDA out of memory. Tried to allocate 2.00 GiB
```

<details><summary>Cause & Fix</summary>

**Cause**: GPU memory exhausted. Could be batch too large, memory leak (storing tensors), or model too big.

**Fix**: Reduce batch size, use gradient checkpointing, mixed precision, or check for `losses.append(loss)` (should be `loss.item()`).
</details>

## 7. Debugging torch.compile

`torch.compile` introduces new error categories: graph breaks, recompilations, and backend failures.

In [ ]:
# Graph break detection with explain()

def fn_with_break(x):
    x = x * 2
    print("debug")  # This causes a graph break!
    x = x + 1
    return x

def fn_clean(x):
    x = x * 2
    x = x + 1
    return x

print("Function WITH graph break:")
dynamo.reset()
exp = dynamo.explain(fn_with_break)(torch.randn(10))
print(f"  Graph breaks: {exp.graph_break_count}")
print(f"  Graphs: {exp.graph_count}")

print("\nFunction WITHOUT graph break:")
dynamo.reset()
exp = dynamo.explain(fn_clean)(torch.randn(10))
print(f"  Graph breaks: {exp.graph_break_count}")
print(f"  Graphs: {exp.graph_count}")

In [ ]:
# Fixing graph breaks

# Pattern 1: Guard debug prints
def fixed_print(x):
    x = x * 2
    if not torch.compiler.is_compiling():
        print("debug")  # Only runs in eager mode
    x = x + 1
    return x

# Pattern 2: Replace data-dependent if with torch.where
def fixed_branch(x):
    # Instead of: if x.sum() > 0: return x*2 else: return x*3
    return torch.where(x.sum() > 0, x * 2, x * 3)

dynamo.reset()
exp1 = dynamo.explain(fixed_print)(torch.randn(10))
dynamo.reset()
exp2 = dynamo.explain(fixed_branch)(torch.randn(10))

print(f"Guarded print: {exp1.graph_break_count} breaks")
print(f"torch.where:   {exp2.graph_break_count} breaks")

In [ ]:
# Recompilation detection with CompileCounter

class CompileCounter:
    def __init__(self):
        self.count = 0
    def __call__(self, gm, example_inputs):
        self.count += 1
        return gm.forward

# Without dynamic shapes: recompiles for each new shape
counter = CompileCounter()
dynamo.reset()

@torch.compile(backend=counter)
def fn(x):
    return x * 2 + 1

print("Without dynamic=True (recompiles for each shape):")
for size in [5, 10, 15, 20]:
    _ = fn(torch.randn(size))
    print(f"  Size {size:>3}: compilations = {counter.count}")

# With dynamic shapes: single compilation
counter2 = CompileCounter()
dynamo.reset()

@torch.compile(backend=counter2, dynamic=True)
def fn2(x):
    return x * 2 + 1

print("\nWith dynamic=True (single compilation):")
for size in [5, 10, 15, 20]:
    _ = fn2(torch.randn(size))
    print(f"  Size {size:>3}: compilations = {counter2.count}")

In [ ]:
# Comparing compiled vs eager (correctness check)

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 5),
        )
    
    def forward(self, x):
        return self.net(x)

# Create two models with identical weights
torch.manual_seed(42)
model_eager = MyModel()

dynamo.reset()
model_compiled = torch.compile(MyModel())
model_compiled.load_state_dict(model_eager.state_dict())

# Compare outputs
x = torch.randn(4, 10)
out_eager = model_eager(x)
out_compiled = model_compiled(x)

max_diff = (out_eager - out_compiled).abs().max().item()
print(f"Max difference: {max_diff:.2e}")
print(f"Outputs match: {torch.allclose(out_eager, out_compiled, atol=1e-5)}")

## 8. Memory Leak Detection

The most common memory leak: storing loss tensors instead of scalars.

In [ ]:
# Memory leak detection
import tracemalloc
import gc

model = nn.Linear(100, 50)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# BAD: storing loss tensors
gc.collect()
tracemalloc.start()

bad_losses = []
for _ in range(100):
    x = torch.randn(32, 100)
    loss = F.mse_loss(model(x), torch.randn(32, 50))
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    bad_losses.append(loss)  # Holds computation graph!

bad_current, bad_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

# GOOD: storing scalars
del bad_losses
gc.collect()
tracemalloc.start()

good_losses = []
for _ in range(100):
    x = torch.randn(32, 100)
    loss = F.mse_loss(model(x), torch.randn(32, 50))
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    good_losses.append(loss.item())  # Just a float!

good_current, good_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"BAD  pattern: peak={bad_peak/1024:.1f} KB")
print(f"GOOD pattern: peak={good_peak/1024:.1f} KB")
print(f"\nMemory saved: {(bad_peak - good_peak)/1024:.1f} KB")
print("Rule: Always use loss.item() when logging!")

## 9. Reproducibility Setup

For filing bug reports and deterministic debugging.

In [ ]:
# Complete reproducibility setup

def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Verify reproducibility
results = []
for trial in range(3):
    set_all_seeds(42)
    model = nn.Linear(10, 5)
    x = torch.randn(3, 10)
    out = model(x)
    results.append(out.detach())

print("Reproducibility check (3 trials with same seed):")
print(f"  Trial 1 == Trial 2: {torch.equal(results[0], results[1])}")
print(f"  Trial 2 == Trial 3: {torch.equal(results[1], results[2])}")

# Environment info for bug reports
import sys, platform
print(f"\nEnvironment for bug report:")
print(f"  Python: {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}")
print(f"  OS: {platform.platform()}")
print(f"  CUDA: {torch.cuda.is_available()}")

## 10. Exercise: Debug a Model with NaN Loss

The model below produces NaN loss after a few training steps. Find and fix the bug.

**Hints**:
- Use the NaN detection hook
- Check the forward pass for numerically unstable operations
- Look at what happens when model outputs have certain ranges

In [ ]:
# EXERCISE: Find and fix the NaN bug

class BuggyClassifier(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )
    
    def forward(self, x):
        logits = self.net(x)
        # BUG: manual softmax + log is numerically unstable
        probs = torch.softmax(logits, dim=-1)
        log_probs = torch.log(probs)  # log(0) = -inf when a prob rounds to 0!
        return log_probs

# Training loop that will produce NaN
torch.manual_seed(0)
model = BuggyClassifier(20, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)  # LR too high!

print("Training with buggy model (high LR + unstable log):")
for step in range(20):
    x = torch.randn(8, 20)
    targets = torch.randint(0, 5, (8,))
    
    log_probs = model(x)
    loss = F.nll_loss(log_probs, targets)
    
    if torch.isnan(loss):
        print(f"  Step {step}: NaN loss detected!")
        break
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    if step % 5 == 0:
        print(f"  Step {step}: loss = {loss.item():.4f}")

In [ ]:
# SOLUTION: Fix the NaN bug

class FixedClassifier(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )
    
    def forward(self, x):
        logits = self.net(x)
        # FIX: Use log_softmax (numerically stable)
        return F.log_softmax(logits, dim=-1)

# Fixed training loop
torch.manual_seed(0)
model = FixedClassifier(20, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # FIX: reasonable LR

print("Training with fixed model:")
for step in range(20):
    x = torch.randn(8, 20)
    targets = torch.randint(0, 5, (8,))
    
    log_probs = model(x)
    loss = F.nll_loss(log_probs, targets)
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    if step % 5 == 0:
        print(f"  Step {step}: loss = {loss.item():.4f}")

print("\nFixes applied:")
print("  1. Replaced log(softmax(x)) with log_softmax(x)")
print("  2. Reduced learning rate from 1.0 to 0.01")

## Key Takeaways

1. **Reproduce first** — Create a minimal script (<20 lines) that triggers the bug
2. **Anomaly detection** — `torch.autograd.detect_anomaly()` catches NaN in backward (debug only, 2-5x overhead)
3. **NaN hooks** — Register forward hooks to catch NaN/Inf as they appear
4. **Gradient flow** — Check `param.grad.norm()` for all layers; vanishing = norm near 0, exploding = norm large
5. **Shape debugging** — Use hooks or model surgery to trace shapes through the network
6. **torch.compile** — Use `explain()` for graph breaks, `CompileCounter` for compilations, `dynamic=True` for varying shapes
7. **Memory** — Use `loss.item()` not `loss`; never store tensors across steps
8. **Reproducibility** — Set all seeds + `deterministic_algorithms` + capture `torch.__config__.show()`

---

**Next steps**: Apply these debugging techniques to your own training pipelines. Whenever loss is NaN or training stalls, come back to this systematic checklist.